# IDSIA nano-drone benchmark

Official split: Square / Random / Chirp runs 1–3 train, run 4 validates, Melon tests. Math is in [`../exploiting_symmetry.md`](../exploiting_symmetry.md).

**Ours** — hist-10 64-64 MLP on the stacked quotient $(g^{\mathrm{body}}, v^{\mathrm{body}}, \omega, \Omega)$, Z$_2\times$Z$_2$ orbit. No GRU.

**Theirs**

1. Physics — published Crazyflie rigid-body step (`omega_dot = 0`)
2. Physics + residual — their hybrid checkpoint


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "quadrotor" / "idsia").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from quadrotor.evaluate import benchmark_mae
from quadrotor.idsia.baselines import load_theirs
from quadrotor.idsia.data import DEFAULT_ROOT, load_idsia_prepared
from quadrotor.fit import n_params
from quadrotor.idsia.history_mlp import load_or_train_headline

DEVICE = "mps"
print("repo", ROOT)
print("IDSIA clone", DEFAULT_ROOT)

In [ ]:
idsia_train, idsia_val, idsia_test = load_idsia_prepared()
print(
    f"train {len(idsia_train)}  val {len(idsia_val)}  test {len(idsia_test)}  "
    f"test {[flight.name for flight in idsia_test]}"
)

Train or load the hist-10 Z2xZ2 MLP. Checkpoint is `runs/idsia_mlp_hist10/` (does not touch `idsia_parity18k`).

In [ ]:
headline = load_or_train_headline(device=DEVICE)
ours = {"hist10": headline}
histories = {}
print(f"ours hist-10 Z2xZ2  params {n_params(headline)}  history {headline.history}")

Load their physics and physics+residual. Physics is analytic. Physics+residual is their published checkpoint (trained here only if that file is missing).

In [ ]:
theirs = load_theirs(idsia_train, idsia_val, device=DEVICE)
for name, model in theirs.items():
    print(f"theirs {name:10s}  params {n_params(model)}")

Training curves for the models we fit. Solid is validation (the checkpoint metric); dashed is train. Physics has no curve.

In [ ]:
import plotly.graph_objects as go

CURVE_LABELS = {
    "hist10": "hist-10 Z2xZ2",
}
CURVE_COLORS = {
    "hist10": "#6d4c9e",
}

fig = go.Figure()
for name, history in histories.items():
    if not history:
        print(f"no history for {name}")
        continue
    epochs = [row["epoch"] for row in history]
    color = CURVE_COLORS[name]
    label = CURVE_LABELS[name]
    fig.add_scatter(
        x=epochs,
        y=[row["val_loss"] for row in history],
        name=f"{label} val",
        mode="lines",
        line=dict(color=color, width=2.2),
        legendgroup=name,
    )
    fig.add_scatter(
        x=epochs,
        y=[row["train_loss"] for row in history],
        name=f"{label} train",
        mode="lines",
        line=dict(color=color, width=1.4, dash="dash"),
        legendgroup=name,
    )
fig.update_yaxes(title_text="standardized full-state MSE", type="log")
fig.update_xaxes(title_text="epoch")
fig.update_layout(
    template="plotly_white",
    title="IDSIA training curves",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    height=480,
    width=900,
    margin=dict(l=56, r=24, t=72, b=48),
)
fig.show()

Open-loop MAE on the whole Melon test set at $h=1$, $h=10$, and $h=50$ (0.01 s, 0.10 s, 0.50 s).

In [ ]:
LABELS = {
    "hist10": "hist-10 Z2xZ2",
    "physics": "physics",
    "phys+res": "physics+residual",
}
HORIZONS = (("h=1", 0), ("h=10", 9), ("h=50", 49))
STATES = (
    ("position", "position", "m"),
    ("velocity", "velocity", "m/s"),
    ("rotation", "rotation", "rad"),
    ("omega", "angular rate", "rad/s"),
)

models = {**ours, **theirs}
rows = []
for name, model in models.items():
    warm = int(getattr(model, "history", 0) or 0)
    metrics = benchmark_mae(model, idsia_test, start_stride=1, warm_steps=warm)
    row = {"model": name}
    for state, _label, _unit in STATES:
        series = metrics[state]
        row[state] = {key: float(series[index]) for key, index in HORIZONS}
    rows.append(row)
    print(f"scored {LABELS[name]}")

print("\nMelon test MAE")
for state, label, unit in STATES:
    print(f"\n{label} [{unit}]")
    print(f"{'model':20s}  {'h=1':>8s}  {'h=10':>8s}  {'h=50':>8s}")
    for row in rows:
        vals = row[state]
        print(
            f"{LABELS[row['model']]:20s}  "
            f"{vals['h=1']:8.4f}  {vals['h=10']:8.4f}  {vals['h=50']:8.4f}"
        )

Open-loop error versus horizon for three 50-step (0.5 s) windows, one on each Melon test flight, all seeded at $t=20\,\mathrm{s}$. Same units as the paper MAE figure: position, velocity, rotation, angular rate. Uses the models already loaded above.

In [ ]:
import numpy as np
import torch
from plotly.subplots import make_subplots

from quadrotor.train import _integrate, _so3_log, module_device, prime_hidden

START_TIME_S = 20.0
ROLL_STEPS = 50
LABELS = {
    "hist10": "hist-10 Z2xZ2",
    "physics": "physics",
    "phys+res": "physics+residual",
}
VIS_COLORS = {
    "hist10": "#6d4c9e",
    "physics": "#5c5c5c",
    "phys+res": "#c4504d",
}


def _start_index(flight, time_s):
    elapsed = np.cumsum(np.concatenate([[0.0], flight.dt]))
    return int(np.searchsorted(elapsed, time_s, side="left"))


def window_mae(model, flight, start, horizon=ROLL_STEPS):
    device = module_device(model)
    stop = start + horizon
    pwm = torch.as_tensor(flight.pwm[start:stop][None], device=device)
    voltage = torch.as_tensor(flight.voltage[start:stop][None], device=device)
    dt = torch.as_tensor(flight.dt[start:stop][None], device=device)
    position = torch.as_tensor(flight.position[start : stop + 1][None], device=device)
    rotation = torch.as_tensor(flight.rotation[start : stop + 1][None], device=device)
    velocity = torch.as_tensor(flight.velocity_world[start : stop + 1][None], device=device)
    omega = torch.as_tensor(flight.omega_body[start : stop + 1][None], device=device)
    model.eval()
    with torch.no_grad():
        warm = int(getattr(model, "history", 0) or 0)
        hidden = prime_hidden(model, [flight], [(0, start)], warm) if warm else None
        position_hat, rotation_hat, velocity_hat, omega_hat = _integrate(
            model,
            pwm,
            dt,
            position[:, 0],
            rotation[:, 0],
            velocity[:, 0],
            omega[:, 0],
            voltage,
            hidden=hidden,
        )
        relative = torch.matmul(rotation[:, 1:].transpose(-1, -2), rotation_hat)
        return {
            "position": torch.linalg.vector_norm(position_hat - position[:, 1:], dim=-1)[0].cpu().numpy(),
            "velocity": torch.linalg.vector_norm(velocity_hat - velocity[:, 1:], dim=-1)[0].cpu().numpy(),
            "rotation": torch.linalg.vector_norm(_so3_log(relative[0]), dim=-1).cpu().numpy(),
            "omega": torch.linalg.vector_norm(omega_hat - omega[:, 1:], dim=-1)[0].cpu().numpy(),
        }


vis_models = {**ours, **theirs}
starts = []
for flight in idsia_test:
    start = _start_index(flight, START_TIME_S)
    if start + ROLL_STEPS >= flight.n_steps:
        raise ValueError(
            f"{flight.name} is too short for a {ROLL_STEPS}-step rollout at t={START_TIME_S}s "
            f"(start {start}, steps {flight.n_steps})"
        )
    starts.append(start)
    print(f"{flight.name}: t={START_TIME_S:.0f}s -> index {start}")

steps = np.arange(1, ROLL_STEPS + 1)
state_titles = (
    ("position", "position MAE [m]"),
    ("velocity", "velocity MAE [m/s]"),
    ("rotation", "rotation MAE [rad]"),
    ("omega", "angular-rate MAE [rad/s]"),
)
fig = make_subplots(
    rows=len(idsia_test),
    cols=4,
    shared_xaxes=True,
    vertical_spacing=0.08,
    horizontal_spacing=0.06,
    subplot_titles=tuple(title for _state, title in state_titles),
    row_titles=tuple(flight.name.replace("melon_20251017_", "") for flight in idsia_test),
)
for row_i, (flight, start) in enumerate(zip(idsia_test, starts), start=1):
    for name, model in vis_models.items():
        errors = window_mae(model, flight, start)
        for col, (state, _title) in enumerate(state_titles, start=1):
            fig.add_scatter(
                x=steps,
                y=errors[state],
                name=LABELS[name],
                mode="lines",
                line=dict(color=VIS_COLORS[name], width=2),
                legendgroup=name,
                showlegend=row_i == 1 and col == 1,
                row=row_i,
                col=col,
            )
    fig.update_xaxes(title_text="h [steps]" if row_i == len(idsia_test) else None, row=row_i)

fig.update_layout(
    template="plotly_white",
    title=f"Open-loop error vs horizon, seed at t = {START_TIME_S:.0f} s",
    legend=dict(orientation="h", yanchor="bottom", y=1.04, x=0),
    height=280 * len(idsia_test),
    width=1100,
    margin=dict(l=80, r=24, t=90, b=48),
)
fig.show()

Same three windows ($t=20\,\mathrm{s}$, 50 steps). The 3D view is Vicon position for a few seconds of context, with the 0.5 s ground-truth segment and each model's open-loop path on top. Charts below are velocity, roll/pitch/yaw, and body rates for that 0.5 s only.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from quadrotor.geometry import roll_pitch_yaw
from quadrotor.train import _integrate, module_device

START_TIME_S = 20.0
ROLL_STEPS = 50
LABELS = {
    "hist10": "hist-10 Z2xZ2",
    "physics": "physics",
    "phys+res": "physics+residual",
}
VIS_COLORS = {
    "hist10": "#6d4c9e",
    "physics": "#5c5c5c",
    "phys+res": "#c4504d",
}

CONTEXT_BEFORE_S = 2.0
CONTEXT_AFTER_S = 5.0


def _time_index(flight, time_s):
    elapsed = np.cumsum(np.concatenate([[0.0], flight.dt]))
    return int(np.clip(np.searchsorted(elapsed, time_s, side="left"), 0, flight.n_steps))


def window_rollout(model, flight, start, horizon=ROLL_STEPS):
    device = module_device(model)
    stop = start + horizon
    pwm = torch.as_tensor(flight.pwm[start:stop][None], device=device)
    voltage = torch.as_tensor(flight.voltage[start:stop][None], device=device)
    dt = torch.as_tensor(flight.dt[start:stop][None], device=device)
    position = torch.as_tensor(flight.position[start : stop + 1][None], device=device)
    rotation = torch.as_tensor(flight.rotation[start : stop + 1][None], device=device)
    velocity = torch.as_tensor(flight.velocity_world[start : stop + 1][None], device=device)
    omega = torch.as_tensor(flight.omega_body[start : stop + 1][None], device=device)
    model.eval()
    with torch.no_grad():
        warm = int(getattr(model, "history", 0) or 0)
        hidden = prime_hidden(model, [flight], [(0, start)], warm) if warm else None
        position_hat, rotation_hat, velocity_hat, omega_hat = _integrate(
            model,
            pwm,
            dt,
            position[:, 0],
            rotation[:, 0],
            velocity[:, 0],
            omega[:, 0],
            voltage,
            hidden=hidden,
        )
    return {
        "position": np.concatenate(
            [flight.position[start : start + 1], position_hat[0].cpu().numpy()], axis=0
        ),
        "velocity": np.concatenate(
            [flight.velocity_world[start : start + 1], velocity_hat[0].cpu().numpy()], axis=0
        ),
        "rotation": np.concatenate(
            [flight.rotation[start : start + 1], rotation_hat[0].cpu().numpy()], axis=0
        ),
        "omega": np.concatenate(
            [flight.omega_body[start : start + 1], omega_hat[0].cpu().numpy()], axis=0
        ),
    }


vis_models = {**ours, **theirs}
windows = []
for flight in idsia_test:
    start = _time_index(flight, START_TIME_S)
    stop = start + ROLL_STEPS
    ctx0 = _time_index(flight, START_TIME_S - CONTEXT_BEFORE_S)
    ctx1 = _time_index(flight, START_TIME_S + CONTEXT_AFTER_S)
    predicted = {name: window_rollout(model, flight, start) for name, model in vis_models.items()}
    windows.append(
        {
            "flight": flight,
            "start": start,
            "stop": stop,
            "ctx0": ctx0,
            "ctx1": ctx1,
            "gt": {
                "position": flight.position[start : stop + 1],
                "velocity": flight.velocity_world[start : stop + 1],
                "rotation": flight.rotation[start : stop + 1],
                "omega": flight.omega_body[start : stop + 1],
            },
            "predicted": predicted,
        }
    )
    print(f"{flight.name}: 3D context [{ctx0}:{ctx1}]  rollout [{start}:{stop}]")

fig = plt.figure(figsize=(13.5, 4.6))
for index, window in enumerate(windows):
    ax = fig.add_subplot(1, 3, index + 1, projection="3d")
    flight = window["flight"]
    context = flight.position[window["ctx0"] : window["ctx1"] + 1]
    ax.plot(context[:, 0], context[:, 1], context[:, 2], color="#c8c8c8", lw=1.2, label="Vicon context")
    gt = window["gt"]["position"]
    ax.plot(gt[:, 0], gt[:, 1], gt[:, 2], color="#222222", lw=2.4, label="Vicon 0.5 s")
    ax.scatter(*gt[0], color="#222222", s=28, depthshade=False)
    for name, pred in window["predicted"].items():
        xyz = pred["position"]
        ax.plot(
            xyz[:, 0],
            xyz[:, 1],
            xyz[:, 2],
            color=VIS_COLORS[name],
            lw=1.8,
            label=LABELS[name],
        )
    ax.set_title(flight.name.replace("melon_20251017_", ""))
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    ax.view_init(elev=22, azim=-60)
    try:
        ax.set_box_aspect(
            np.ptp(context, axis=0) + 1e-6
        )
    except TypeError:
        pass
    if index == 0:
        ax.legend(loc="upper left", fontsize=8)
fig.suptitle(f"Position, seed at t = {START_TIME_S:.0f} s", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

time_s = np.arange(ROLL_STEPS + 1) * 0.01
xyz_labels = ("x [m]", "y [m]", "z [m]")
fig = make_subplots(
    rows=len(windows),
    cols=3,
    shared_xaxes=True,
    vertical_spacing=0.08,
    horizontal_spacing=0.06,
    subplot_titles=xyz_labels,
    row_titles=tuple(
        window["flight"].name.replace("melon_20251017_", "") for window in windows
    ),
)
for row, window in enumerate(windows, start=1):
    gt = window["gt"]["position"]
    for col in range(3):
        fig.add_scatter(
            x=time_s,
            y=gt[:, col],
            name="Vicon",
            mode="lines",
            line=dict(color="#222222", width=2.2, dash="dash"),
            legendgroup="vicon",
            showlegend=row == 1 and col == 0,
            row=row,
            col=col + 1,
        )
        for name, pred in window["predicted"].items():
            fig.add_scatter(
                x=time_s,
                y=pred["position"][:, col],
                name=LABELS[name],
                mode="lines",
                line=dict(color=VIS_COLORS[name], width=1.7),
                legendgroup=name,
                showlegend=row == 1 and col == 0,
                row=row,
                col=col + 1,
            )
        fig.update_yaxes(title_text=xyz_labels[col] if row == 1 else None, row=row, col=col + 1)
for col in range(1, 4):
    fig.update_xaxes(title_text="t [s] from seed", row=len(windows), col=col)

fig.update_layout(
    template="plotly_white",
    title=f"Position x, y, z — 0.5 s open-loop from t = {START_TIME_S:.0f} s",
    legend=dict(orientation="h", yanchor="bottom", y=1.06, x=0),
    height=280 * len(windows),
    width=1100,
    margin=dict(l=72, r=24, t=90, b=48),
)
fig.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

time_s = np.arange(ROLL_STEPS + 1) * 0.01
state_rows = (
    ("velocity", None, ("v<sub>x</sub> [m/s]", "v<sub>y</sub> [m/s]", "v<sub>z</sub> [m/s]")),
    ("rotation", "rpy", ("roll φ [rad]", "pitch θ [rad]", "yaw ψ [rad]")),
    ("omega", None, ("ω<sub>x</sub> [rad/s]", "ω<sub>y</sub> [rad/s]", "ω<sub>z</sub> [rad/s]")),
)

for window in windows:
    flight_name = window["flight"].name.replace("melon_20251017_", "")
    fig = make_subplots(
        rows=3,
        cols=3,
        shared_xaxes=True,
        vertical_spacing=0.12,
        horizontal_spacing=0.08,
        subplot_titles=tuple(label for _key, _kind, labels in state_rows for label in labels),
    )
    for row, (key, kind, _labels) in enumerate(state_rows, start=1):
        gt = window["gt"][key]
        if kind == "rpy":
            gt = roll_pitch_yaw(gt)
        for col in range(3):
            fig.add_scatter(
                x=time_s,
                y=gt[:, col],
                name="Vicon",
                mode="lines",
                line=dict(color="#222222", width=2.4, dash="dash"),
                legendgroup="vicon",
                showlegend=row == 1 and col == 0,
                row=row,
                col=col + 1,
            )
            for name, pred in window["predicted"].items():
                series = pred[key]
                if kind == "rpy":
                    series = roll_pitch_yaw(series)
                fig.add_scatter(
                    x=time_s,
                    y=series[:, col],
                    name=LABELS[name],
                    mode="lines",
                    line=dict(color=VIS_COLORS[name], width=2),
                    legendgroup=name,
                    showlegend=row == 1 and col == 0,
                    row=row,
                    col=col + 1,
                )
            fig.update_xaxes(title_text="t [s] from seed", row=3, col=col + 1)
    fig.update_yaxes(title_font=dict(size=13), tickfont=dict(size=12))
    fig.update_xaxes(tickfont=dict(size=12))
    fig.update_annotations(font_size=14)
    fig.update_layout(
        template="plotly_white",
        title=f"{flight_name}: velocity, attitude, body rates from t = {START_TIME_S:.0f} s",
        legend=dict(orientation="h", yanchor="bottom", y=1.08, x=0, font=dict(size=12)),
        height=820,
        width=1100,
        margin=dict(l=64, r=28, t=110, b=56),
    )
    fig.show()

Translation test on their physics+residual. The residual MLP reads the full world state $(x,y,z,\ldots)$, so a rigid $+2\,\mathrm{m}$ shift in $x$ and $y$ should change the leftover. Ground truth is shifted by the same amount: an SE(3)-invariant plant would keep the same MAE.

In [ ]:
from dataclasses import replace

import numpy as np
from quadrotor.evaluate import benchmark_mae
from quadrotor.idsia.baselines import load_theirs

SHIFT_XY = np.array([2.0, 2.0, 0.0], dtype=np.float32)
HORIZONS = (("h=1", 0), ("h=10", 9), ("h=50", 49))
STATES = (
    ("position", "position", "m"),
    ("velocity", "velocity", "m/s"),
    ("rotation", "rotation", "rad"),
    ("omega", "angular rate", "rad/s"),
)


def shift_xy(flight, delta=SHIFT_XY):
    return replace(flight, position=np.asarray(flight.position, dtype=np.float32) + delta)


if "theirs" not in globals() or "phys+res" not in theirs:
    theirs = load_theirs(idsia_train, idsia_val, device=DEVICE)
if "idsia_test" not in globals():
    idsia_train, idsia_val, idsia_test = load_idsia_prepared()

phys_res = theirs["phys+res"]
shifted_test = [shift_xy(flight) for flight in idsia_test]
original = benchmark_mae(phys_res, idsia_test, start_stride=1)
perturbed = benchmark_mae(phys_res, shifted_test, start_stride=1)

print("physics+residual  Melon MAE  original vs +2 m in x and y")
print(f"{'state':16s}  {'orig h=1':>9s}  {'+2 m':>9s}  {'orig h=10':>9s}  {'+2 m':>9s}  {'orig h=50':>9s}  {'+2 m':>9s}")
for state, label, unit in STATES:
    a, b = original[state], perturbed[state]
    print(
        f"{label:16s}  {a[0]:9.4f}  {b[0]:9.4f}  {a[9]:9.4f}  {b[9]:9.4f}  {a[49]:9.4f}  {b[49]:9.4f}  [{unit}]"
    )
    print(
        f"{'  Δ MAE':16s}  {b[0] - a[0]:+9.4f}  {'':9s}  {b[9] - a[9]:+9.4f}  {'':9s}  {b[49] - a[49]:+9.4f}"
    )

First Melon flight, seed at $t=20\,\mathrm{s}$, 0.5 s open loop. Left is the original window (Vicon context, Vicon 0.5 s, physics+residual). Right is the same motors after $+2\,\mathrm{m}$ in $x$ and $y$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from quadrotor.train import _integrate, module_device

START_TIME_S = 20.0
ROLL_STEPS = 50
CONTEXT_BEFORE_S = 0.5
CONTEXT_AFTER_S = 0.5
PHYS_RES_COLOR = "#c4504d"


def _time_index(flight, time_s):
    elapsed = np.cumsum(np.concatenate([[0.0], flight.dt]))
    return int(np.clip(np.searchsorted(elapsed, time_s, side="left"), 0, flight.n_steps))


def window_rollout(model, flight, start, horizon=ROLL_STEPS):
    device = module_device(model)
    stop = start + horizon
    pwm = torch.as_tensor(flight.pwm[start:stop][None], device=device)
    voltage = torch.as_tensor(flight.voltage[start:stop][None], device=device)
    dt = torch.as_tensor(flight.dt[start:stop][None], device=device)
    position = torch.as_tensor(flight.position[start : stop + 1][None], device=device)
    rotation = torch.as_tensor(flight.rotation[start : stop + 1][None], device=device)
    velocity = torch.as_tensor(flight.velocity_world[start : stop + 1][None], device=device)
    omega = torch.as_tensor(flight.omega_body[start : stop + 1][None], device=device)
    model.eval()
    with torch.no_grad():
        position_hat, _rotation_hat, _velocity_hat, _omega_hat = _integrate(
            model,
            pwm,
            dt,
            position[:, 0],
            rotation[:, 0],
            velocity[:, 0],
            omega[:, 0],
            voltage,
        )
    return np.concatenate([flight.position[start : start + 1], position_hat[0].cpu().numpy()], axis=0)


flight = idsia_test[0]
shifted = shift_xy(flight)
start = _time_index(flight, START_TIME_S)
stop = start + ROLL_STEPS
ctx0 = _time_index(flight, START_TIME_S - CONTEXT_BEFORE_S)
ctx1 = _time_index(flight, START_TIME_S + CONTEXT_AFTER_S)
print(f"{flight.name}: t={START_TIME_S:.0f}s  index {start}  context [{ctx0}:{ctx1}]")

original_path = window_rollout(phys_res, flight, start)
perturbed_path = window_rollout(phys_res, shifted, start)

panels = (
    ("original", flight, original_path),
    ("+2 m in x and y", shifted, perturbed_path),
)
fig = plt.figure(figsize=(12.4, 5.4))
for index, (title, shown, pred) in enumerate(panels):
    ax = fig.add_subplot(1, 2, index + 1, projection="3d")
    context = shown.position[ctx0 : ctx1 + 1]
    gt = shown.position[start : stop + 1]
    ax.plot(context[:, 0], context[:, 1], context[:, 2], color="#c8c8c8", lw=1.2, label="Vicon context")
    ax.plot(gt[:, 0], gt[:, 1], gt[:, 2], color="#222222", lw=2.4, label="Vicon 0.5 s")
    ax.scatter(*gt[0], color="#222222", s=28, depthshade=False)
    ax.plot(pred[:, 0], pred[:, 1], pred[:, 2], color=PHYS_RES_COLOR, lw=1.8, label="physics+residual")
    ax.set_title(title)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    ax.view_init(elev=22, azim=-60)
    try:
        ax.set_box_aspect(np.ptp(np.vstack([context, pred]), axis=0) + 1e-6)
    except TypeError:
        pass
    if index == 0:
        ax.legend(loc="upper left", fontsize=8)
fig.suptitle(f"{flight.name}  seed at t = {START_TIME_S:.0f} s", y=1.02)
fig.tight_layout()
plt.show()

orig_err = np.linalg.norm(original_path - flight.position[start : stop + 1], axis=-1)
pert_err = np.linalg.norm(perturbed_path - shifted.position[start : stop + 1], axis=-1)
print(f"this window  position RMSE  original {orig_err.mean():.4f} m  +2 m xy {pert_err.mean():.4f} m")
print(f"this window  position at h=50  original {orig_err[-1]:.4f} m  +2 m xy {pert_err[-1]:.4f} m")